# Importing libraries

In [ ]:
from ctypes import kind
from unicodedata import category
from unittest.mock import inplace

import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

# Importing CSVs

## Import soldproducts clean

In [ ]:
# Import csvs

sold_products_df = pd.read_csv('/Users/mathiasvandenbossche/Desktop/📊 DSAI 007/ENIAC PROJECT 2/soldproducts_clean.csv')

In [ ]:
sold_products_df.info()

## Import products not sold

In [ ]:
# Import csvs

notsold_df = pd.read_csv('/Users/mathiasvandenbossche/Desktop/📊 DSAI 007/ENIAC PROJECT 2/notsold_df.csv')

## import orderlines

In [ ]:
# Import csvs

orderlines_df = pd.read_csv('/Users/mathiasvandenbossche/Desktop/📊 DSAI 007/ENIAC PROJECT 2/orderlines_clean.csv')

## Import orders

In [ ]:
# Import csvs

orders_df = pd.read_csv('/Users/mathiasvandenbossche/Desktop/📊 DSAI 007/ENIAC PROJECT 2/orders_clean.csv')

In [ ]:
orders_df.info()

# Which orders?

for this project, the most relevant and accurate data would be the completed orders only.
Lets seperate them.

## Order distribution

In [ ]:
# Lets check the distribution of different order states in percentage.

orders_df['state'].value_counts(normalize=True) *100



In [ ]:
# Completed orders will be the most trustworthy for the scope of our project.
# Total rows: 226904
# Completed orders rows: 46605
# Percentage completed orders rows: 21.0%

total_rows = orders_df['order_id'].count()
Completed_count_rows = orders_df['order_id'][orders_df['state'] == 'Completed'].count()
percentage_completed_rows = round(Completed_count_rows/total_rows,2) *100

print(f'Total rows: {total_rows}')
print(f'Completed rows: {Completed_count_rows}')
print(f'Percentage completed rows: {percentage_completed_rows}%')

## Completed orders

In [ ]:
# Saving all the completed order rows under a new variable.

completed_orders = orders_df.loc[orders_df['state'] == 'Completed']

In [ ]:
completed_orders.info()

right, now we have the completed rows. We will have to apply this to the orderlines as well. We can only use rows in orderlines that are part of orders that were completed. I'm gonna left join orderlines with completed orders on order ID and remove Null values after.

In [ ]:
# I just noticed that the order_id name in orderlines is reversed. I could use left_on= and right_on= but I prefer to just rename that column for consistency.

orderlines_df.rename(columns={'id_order': 'order_id'}, inplace=True)

orderlines_df.info()



In [ ]:
# With the name changed, I now merged the completed orders with orderlines.


all_orderlines_completed_and_null = orderlines_df.merge(completed_orders, on= 'order_id',  how='left')

In [ ]:
all_orderlines_completed_and_null.info()

In [ ]:
# I'm making a mask where I filter for only the rows without null values. These orderlines are what we will use for our analysis.

orderlines_completed_orders_noNAN = all_orderlines_completed_and_null[all_orderlines_completed_and_null.notna().all(axis=1)]
orderlines_completed_orders_noNAN.info()




In [ ]:
# As seen above, we are trimming a significant amount of orderlines out.
# below an overview again:

total_rows_orderlines = all_orderlines_completed_and_null['order_id'].shape[0]
no_NaN_orderlines = orderlines_completed_orders_noNAN['order_id'].shape[0]
percentage_completed_rows = round(no_NaN_orderlines/total_rows_orderlines,2) *100

print(f'Total rows: {total_rows_orderlines}')
print(f'orderlines from completed orders: {no_NaN_orderlines}')
print(f'Percentage of orders_lines that are connected to completed orders: {percentage_completed_rows}%')

# This is exactly the same percentage as completed orders versus non-completed. this means our merge was a success!


## Rest of orders

In [ ]:
# these are all the ordersLINES that are in the OTHER ORDER STATE categories

not_completed_orderlines = all_orderlines_completed_and_null[all_orderlines_completed_and_null.isna().any(axis=1)]

not_completed_orderlines.info()

In [ ]:
### dropping the null columns , saving under checkpoint var.

nc_orderlines_CP = not_completed_orderlines.drop(columns=['created_date', 'total_paid', 'state'])

In [ ]:
## successfully dropped.

nc_orderlines_CP.info()

In [ ]:
# these are all the ORDERS that are NOT COMPLETED


all_other_orders = orders_df.loc[orders_df['state'] != 'Completed']


In [ ]:
# Here i merge the orderlines with all the other categories.

Orderlines_Out_of_Scope = nc_orderlines_CP.merge(all_other_orders, how='left', on='order_id').copy()

In [ ]:
Orderlines_Out_of_Scope.info()

## DF In-scope Orderlines

In [ ]:
## Making a copy and saving it under the final variable

Orderslines_inscope = orderlines_completed_orders_noNAN.copy()

In [ ]:
Orderslines_inscope.info()

## DF Out-Of-Scope Orderlines

In [ ]:
# Same for the reverse, here is our final with non completed Orderlines.

In [ ]:
Orderlines_Out_of_Scope.info()

check if all the skus exist in products.

In [ ]:
## Merging the Orderlines with the products under a throw-away variable.

_ = Orderslines_inscope.merge(sold_products_df, on='sku',  how='left')

In [ ]:
## Checking which products are not known in the resulting DF. Only 0.6% is not known. that is good news. We have to remove these but we have to remove all the orders from that as well.

_['name'].isna().value_counts( normalize= True) *100

In [ ]:
# I'm going to extract the order_ids from these orders.
# as a set can not continue unique values. I prefer this over chaining .unique().

orders_to_delete = set(_.loc[_['name'].isna(), 'order_id'])
orders_to_delete

In [ ]:
len(orders_to_delete)

In [ ]:
# Dropping all orders-lines from orders that are in the above set.

Orderslines_inscope = Orderslines_inscope.loc[~Orderslines_inscope['order_id'].isin(orders_to_delete)]

# Paid vs Price

In [ ]:
## Lets first drop the 'ID' Column as we don't really need that. We have an index that is unique.

WIP_c_orders = Orderslines_inscope.drop('id', axis=1).copy()



In [ ]:
# Calculate the total price based on quantity * unit_price

WIP_c_orders['total_unit_price'] = WIP_c_orders.unit_price * WIP_c_orders.product_quantity

In [ ]:
# We are going to compare the price of sum the units per order. so essentially calculate the raw total of price of the products grouped per order. We are going to compare that with what the customer has paid. to see if there is a big difference.

Price_vs_Paid = WIP_c_orders.groupby('order_id').agg( sum_total_unit_price = ('total_unit_price', 'sum'), order_paid = ( 'total_paid' , 'max') ).reset_index()

In [ ]:
Price_vs_Paid['Paid_difference']= Price_vs_Paid['order_paid'] - Price_vs_Paid['sum_total_unit_price']

In [ ]:
Price_vs_Paid['Paid_difference'].plot( kind= 'hist', figsize= (10,5), grid= True)


In [ ]:
ax = Price_vs_Paid.boxplot("Paid_difference");
ax.set_ylim(-50, +100)
plt.show()

In [ ]:
Price_vs_Paid.Paid_difference.describe()

## Outliers and IQR

lets calculate the IQR: We do so by deducting the 75% (Q3) from the 25% (Q1)  The IQR represents the middle 50% of ourdata.

In [ ]:
Q1 = Price_vs_Paid.Paid_difference.quantile(0.25)
Q3 = Price_vs_Paid.Paid_difference.quantile(0.75)
Q1, Q3

In [ ]:
IQR = Q3 - Q1
IQR

In [ ]:
# Now that we have the middle 50% of our data we have to extend that range somehow. the standard is to extend the boundaries with 1.5 * IQR for the upper and the lower boundary.

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)
lower_bound, upper_bound

Perfect, we have a threshold for our outliers. Lets filter this to our Price_vs_Paid, extract the ordernumbers, and filter our main DF and take out all the order IDs that are considered outliers.

In [ ]:
## Applying the boundaries to our price vs paid frame.

no_outliers = Price_vs_Paid.loc[(Price_vs_Paid['Paid_difference']>= lower_bound) & (Price_vs_Paid['Paid_difference']<= upper_bound)]

In [ ]:
orders_in_range = set(no_outliers['order_id'])


In [ ]:
no_outliers.info()

In [ ]:
print(f'there are {len(orders_in_range)} Order IDs that are NOT concidered outliers.')
print(f"We have {len(set(WIP_c_orders['order_id']))} unique order ids in our total df")
print(f"We will remove {len(set(WIP_c_orders['order_id'])) - len(set(orders_in_range))} outliers.")

In [ ]:
# Before i apply this to the main DF ill take a look at the boxplot.

no_outliers['Paid_difference'].plot(kind= 'box', figsize= (10,5), grid= True)
plt.show()

# Looks good!

In [ ]:
#savefile = WIP_c_orders.copy()

In [ ]:
# WIP_c_orders = savefile

In [ ]:
# Im going to remove all the order ID from the main DF.


WIP_c_orders = WIP_c_orders.loc[WIP_c_orders['order_id'].isin(orders_in_range)]




In [ ]:
WIP_c_orders.nunique()

# perfect, we have the exact same number of unique orders in ou DF as in the no_outliers calculation above.

order_id            45407
product_id              1
product_quantity       27
sku                  5911
unit_price           5777
date                59498
created_date        45314
total_paid          12234
state                   1
total_unit_price     7031
dtype: int64

In [ ]:
WIP_c_orders_SAVE =  WIP_c_orders.copy()


In [ ]:
# Extracting the current clean order + orderlines for colleagues.

WIP_c_orders.to_csv('Order_and_orderlines_no_outliers.csv' ,index= False)

In [ ]:
WIP_c_orders.info()

In [ ]:
master_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60163 entries, 0 to 60162
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            60163 non-null  int64  
 1   product_id          60163 non-null  int64  
 2   product_quantity    60163 non-null  int64  
 3   sku                 60163 non-null  object 
 4   unit_price          60163 non-null  float64
 5   date                60163 non-null  object 
 6   created_date        60163 non-null  object 
 7   total_paid          60163 non-null  float64
 8   state               60163 non-null  object 
 9   total_unit_price    60163 non-null  float64
 10  name                60163 non-null  object 
 11  corrected_price     60163 non-null  float64
 12  lowest_sales_price  60163 non-null  float64
 13  type                60163 non-null  object 
 14  in_stock            60163 non-null  int64  
 15  desc                60159 non-null  object 
 16  OLD_

In [ ]:
# Dropping columns not needed currently or will join again with updated info from the products frame.

master_orders_WIP = master_orders.drop(columns=['product_id', 'total_paid','state', 'lowest_sales_price', 'type', 'in_stock', 'OLD_price_OLD', 'name', 'corrected_price', 'desc']).copy()

In [ ]:
# Re-organizing the columns for better readability of the frame.

master_orders_WIP = master_orders_WIP[['order_id', 'date', 'sku', 'unit_price', 'product_quantity', 'total_unit_price']].copy()

In [ ]:
## Renaming the price columns as part of readability

master_orders_WIP.rename(columns={'unit_price': 'selling_price',
                                  'total_unit_price': 'line_total'}, inplace=True)

In [ ]:
master_orders_WIP

,order_id,date,sku,selling_price,product_quantity,line_total
0,299545,2017-01-01 01:46:16,OWC0100,47.49,1,47.49
1,299546,2017-01-01 01:50:34,IOT0014,18.99,1,18.99
2,295347,2017-01-01 01:54:11,APP0700,72.19,1,72.19
3,299549,2017-01-01 02:07:42,PAC0929,2565.99,1,2565.99
4,299556,2017-01-01 02:20:14,CRU0039-A,60.90,1,60.90
...,...,...,...,...,...,...
60158,527035,2018-03-14 11:42:41,APP0698,9.99,1,9.99
60159,527070,2018-03-14 11:49:01,APP0698,9.99,2,19.98
60160,527074,2018-03-14 11:49:36,APP0698,9.99,2,19.98
60161,527096,2018-03-14 11:54:35,APP0698,9.99,3,29.97


# Categories

In [ ]:
sold_products_df['type'].nunique()


# We have 123 different types which can be converted to categories.
# So we can take a look at the most common categories first and how many % of the products this contains.



In [ ]:
# Making a list of all the product names within one category.

typelist = sold_products_df.groupby('type')['name'].apply(list).reset_index()


In [ ]:
# using a function to count the most common words in the that list. We are separating the names again in the list.  then we are finding all the individual words with regex. I initially used .split() but this was returning also a lot of characters or weird prefixes and artifacts.

def find_keywords(row):
    import re
    from collections import Counter
    name = str(row['name']).lower().split(',')
    words = re.findall(r'\w+',str(name))
    ignore_words = {'open','air', 'pro'}
    cleaned_words = [word for word in words if word not in ignore_words]
    counts = Counter(cleaned_words)
    return ' , '.join([x[0] for x in counts.most_common(10)])



In [ ]:
# apply the function to the same frame

typelist['keyword_list'] = typelist.apply(find_keywords, axis=1)

In [ ]:
# result below. We now have all our unique 'type' identifiers with a list of the most common keywords in all the names that are under that type. We can now feed this list to our conditions for arranging them by category buckets.

typelist



In [ ]:
## Create our buckets and filters for the categorisation of our types based on the most common words in all the product names regarding that type.


condlist = [typelist['keyword_list'].str.contains('Repair|Service|Installation|Part|Piece', case=False, na=False),
            typelist['keyword_list'].str.contains('Keyboard|Mouse|Trackpad|Wacom|Stylus|Pencil|Digitizer|Tablet|Microphone|Keypad|Multimedia|gloves', case=False, na=False),
            typelist['keyword_list'].str.contains('Monitor|Headset|Headphones|Speaker|Earpods', case=False, na=False),
            typelist['keyword_list'].str.contains('Case|Protect|Sleeve|Shell|Bag|Cover|Screen Protector|Battery|Strap|Cleaner|Security|Lock|Housing|Watch|Waterproof|Casing|Impact|Support|Bags|Shoulder|Cable|Adapter|HDMI|Hub|Dock|Lightning|Charger|Power|Backpack', case=False, na=False),
            typelist['keyword_list'].str.contains('SATA|SSD|HDD|Drive|Flash Drive|Memory|NAS|RAID|Disk|Cruzer|G-DRIVE|Mercury|Expansion|Hard', case=False, na=False),
            typelist['keyword_list'].str.contains('iMac|MacBook|Mac mini|Mac Pro', case=False, na=False),
            typelist['keyword_list'].str.contains('iPhone|iPad|iPod', case=False, na=False)]

choicelist = [  'Repair / Services',
                'Input devices',
                'Output devices',
                'Accessoires',
                'Memory & storages',
                'Computer',
                'Mobile Phone']

typelist['category'] = np.select(condlist, choicelist, default='Others')

In [ ]:
## We are going to extract the 2 columns and pair each value to the correct category in a dictionary.

cat_map = dict(zip(typelist.type, typelist.category))

words_map = dict(zip(typelist.type, typelist.keyword_list))

In [ ]:
# So for each row we want to look at the type and return the corresponding value from our category map dictionary.

sold_products_df['Category'] = sold_products_df['type'].map(cat_map)

sold_products_df['keywords'] = sold_products_df['type'].map(words_map)

In [ ]:
# Count distribution across our categoriess

sold_products_df.Category.value_counts()

Category
Accessoires          3200
Memory & storages    2371
Input devices         608
Mobile Phone          462
Output devices        429
Repair / Services     284
Computer              282
Others                175
Name: count, dtype: int64

In [ ]:
# Products in Others (175 SKUs), are mostly internet or Ethernet stuff. We could eventually make another category for this. But i'll leave it for now as I have to move on due to limited time.

sold_products_df[sold_products_df['Category'] == 'Others']

,sku,name,Category,corrected_price,lowest_sales_price,type,in_stock,desc,OLD_price_OLD,keywords
10,APP0111,Apple USB Ethernet Adapter Mac,Others,35.00,29.69,1334,0,USB to Ethernet adapter Mac.,35,"link , d , wi , fi , gigabit , tp , router , s..."
69,APP0490,Apple Airport Express,Others,109.00,82.64,1334,0,Airport Express base station 802.11a / b / g / n.,109,"link , d , wi , fi , gigabit , tp , router , s..."
77,DLK0008,D-Link DCS-932L Wireless IP Camera Cloud,Others,79.99,43.19,9094,0,Wireless network camera day and night home wit...,79.99,"camera , surveillance , d , link , dcs , video..."
78,DLK0009,D-Link DCS-942L Wireless IP Camera Cloud,Others,169.99,78.29,9094,0,security camera with Wi-Fi or Ethernet night v...,169.99,"camera , surveillance , d , link , dcs , video..."
142,DLK0018,D-Link DES-1005D Switch 5 10 / 100Mbps,Others,14.99,10.79,1334,0,Dlink Switch 5 Mac and PC ports.,14.99,"link , d , wi , fi , gigabit , tp , router , s..."
...,...,...,...,...,...,...,...,...,...,...
7753,UBI0007,Ubiquiti Amplifi Wi-Fi Mesh Router + 2 Mesh Ac...,Others,359.99,359.99,1334,0,Wi-Fi high-density intelligent Mesh technology,35.998.952,"link , d , wi , fi , gigabit , tp , router , s..."
7754,RIN0018,Ring Chime Pro Extender Wi-Fi and Timbre,Others,59.00,59.00,1334,0,Timbre increases your Wi-Fi signal and amplifi...,59,"link , d , wi , fi , gigabit , tp , router , s..."
7755,UBI0008,Ubiquiti Amplifi Wi-Fi Mesh Router,Others,129.99,129.99,1334,0,Smart Wi-Fi Router high density Mesh technology,1.299.891,"link , d , wi , fi , gigabit , tp , router , s..."
7759,RIN0015,Video surveillance camera Floodlight Ring with...,Others,299.00,299.00,9094,0,Surveillance camera with face detection alarm ...,299,"camera , surveillance , d , link , dcs , video..."


# Merging Orders with Products

In [ ]:
## I'm going to remove columns that are currently no benefit to us.

sold_products_df.drop(columns=['lowest_sales_price', 'type', 'in_stock', 'OLD_price_OLD', 'keywords'], inplace=True)

In [ ]:
## Drop keywords as well

sold_products_df.drop(columns=['keywords'], inplace=True)

In [ ]:
## rename correcrted price to a better name before merging with orders_orderlines

sold_products_df.rename(columns={'corrected_price': 'catalog_price'}, inplace=True)

In [ ]:
sold_products_df

,sku,name,Category,catalog_price,desc
0,RAI0007,Silver Rain Design mStand Support,Accessoires,59.99,Aluminum support compatible with all MacBook
1,APP0023,Apple Mac Keyboard Keypad Spanish,Input devices,59.00,USB ultrathin keyboard Apple Mac Spanish.
2,APP0025,Mighty Mouse Apple Mouse for Mac,Input devices,59.00,mouse Apple USB cable.
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,Accessoires,25.00,IPhone dock and USB Cable Apple iPod.
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,Memory & storages,34.99,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...
...,...,...,...,...,...
7806,MMW0012,"My MW Case MacBook Pro 13 ""(Late 2016) Blue Sa...",Accessoires,29.99,Avoid shock and damage to your MacBook Pro 13-...
7807,MMW0013,"My MW Case MacBook Pro 13 ""(Late 2016) White S...",Accessoires,29.99,Avoid shock and damage to your MacBook Pro 13-...
7808,MMW0016,"My MW Case MacBook Pro 13 ""(Late 2016) Black",Accessoires,29.99,Avoid shock and damage to your MacBook Pro 13-...
7809,MMW0015,"My MW Case MacBook Pro 13 ""(Late 2016) Gray",Accessoires,29.99,Avoid shock and damage to your MacBook Pro 13-...


In [ ]:
master_orders_WIP

,order_id,date,sku,selling_price,product_quantity,line_total
0,299545,2017-01-01 01:46:16,OWC0100,47.49,1,47.49
1,299546,2017-01-01 01:50:34,IOT0014,18.99,1,18.99
2,295347,2017-01-01 01:54:11,APP0700,72.19,1,72.19
3,299549,2017-01-01 02:07:42,PAC0929,2565.99,1,2565.99
4,299556,2017-01-01 02:20:14,CRU0039-A,60.90,1,60.90
...,...,...,...,...,...,...
60158,527035,2018-03-14 11:42:41,APP0698,9.99,1,9.99
60159,527070,2018-03-14 11:49:01,APP0698,9.99,2,19.98
60160,527074,2018-03-14 11:49:36,APP0698,9.99,2,19.98
60161,527096,2018-03-14 11:54:35,APP0698,9.99,3,29.97


In [ ]:
# Merging the orders with the products.
master_orders_full = master_orders_WIP.merge(sold_products_df[['sku', 'name', 'Category', 'catalog_price']], on='sku', how='left')
re_order = ['date', 'sku','name' ,'Category', 'catalog_price',  'selling_price', 'product_quantity', 'line_total']
master_orders_full = master_orders_full[re_order]


In [ ]:
master_orders_full

,date,sku,name,Category,catalog_price,selling_price,product_quantity,line_total
0,2017-01-01 01:46:16,OWC0100,OWC In-line Digital Temperature Sensor Kit HDD...,Accessoires,60.99,47.49,1,47.49
1,2017-01-01 01:50:34,IOT0014,iOttie Easy View 2 Car Black Support,Accessoires,22.95,18.99,1,18.99
2,2017-01-01 01:54:11,APP0700,Apple 85W MagSafe 2 charger MacBook Pro screen...,Accessoires,89.00,72.19,1,72.19
3,2017-01-01 02:07:42,PAC0929,"Apple iMac 27 ""Core i5 3.2GHz Retina 5K | 32GB...",Computer,3209.00,2565.99,1,2565.99
4,2017-01-01 02:20:14,CRU0039-A,(Open) Crucial 240GB SSD 7mm BX200,Accessoires,76.99,60.90,1,60.90
...,...,...,...,...,...,...,...,...
60158,2018-03-14 11:42:41,APP0698,Apple Lightning Cable Connector to USB 1m Whit...,Accessoires,25.00,9.99,1,9.99
60159,2018-03-14 11:49:01,APP0698,Apple Lightning Cable Connector to USB 1m Whit...,Accessoires,25.00,9.99,2,19.98
60160,2018-03-14 11:49:36,APP0698,Apple Lightning Cable Connector to USB 1m Whit...,Accessoires,25.00,9.99,2,19.98
60161,2018-03-14 11:54:35,APP0698,Apple Lightning Cable Connector to USB 1m Whit...,Accessoires,25.00,9.99,3,29.97


In [ ]:
master_orders_full.to_csv('master_orders_full.csv', index=False)

In [ ]:
WIP_c_orders